In [1]:
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers 
import numpy as np 
import matplotlib.pyplot as plt 


In [2]:
(x_train,_), (_,_) = keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') / 255.0 

x_train = np.reshape(x_train, (-1,28,28,1))

x_train.shape

(60000, 28, 28, 1)

In [ ]:
# Build Generator

latent_dim = 100 

generator = keras.Sequential([
    layers.Input(shape=(latent_dim,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(28*28, activation='sigmoid'),
    layers.Reshape((28, 28, 1))
])

generator.summary() 

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               12928     
                                                                 
 dense_1 (Dense)             (None, 784)               101136    
                                                                 
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
Total params: 114,064
Trainable params: 114,064
Non-trainable params: 0
_________________________________________________________________


In [ ]:
#build dicriminator
discriminator = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

discriminator.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense_2 (Dense)             (None, 1288)              1011080   
                                                                 
 dense_3 (Dense)             (None, 1)                 1289      
                                                                 
Total params: 1,012,369
Trainable params: 1,012,369
Non-trainable params: 0
_________________________________________________________________


In [5]:
#Compile discriminator 
discriminator.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.0002),
    loss = 'binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
discriminator.trainable = False 

gan_input = keras.Input(shape=(latent_dim,)) 

#generate fake images
fake_image = generator(gan_input) 

#ask discriminator whether fake image is real
gan_output = discriminator(fake_image)

#combine model
gan = keras.Model(gan_input, gan_output)

gan.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0002),
    loss='binary_crossentropy'
)

gan.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 100)]             0         
                                                                 
 sequential (Sequential)     (None, 28, 28, 1)         114064    
                                                                 
 sequential_1 (Sequential)   (None, 1)                 1012369   
                                                                 
Total params: 1,126,433
Trainable params: 114,064
Non-trainable params: 1,012,369
_________________________________________________________________


In [9]:
#Training
batch_size = 128
epochs = 10000 

for epoch in range(epochs):

    #Train Discriminator
    idx = np.random.randint(0, x_train.shape[0], batch_size)
    real_images = x_train[idx]

    noise = np.random.normal(0,1, (batch_size, latent_dim))
    fake_image = generator.predict(noise, verbose=0) 

    d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size, 1)))
    d_loss_fake = discriminator.train_on_batch(fake_image, np.zeros((batch_size, 1)))

    #Train Generator
    noise = np.random.normal(0,1, (batch_size,latent_dim))
    g_loss = gan.train_on_batch(noise, np.ones((batch_size,1))) 

    if epochs % 1000 == 0:
        print(f"Epoch: {epoch} | "
              f"D_loss: {d_loss_real[0]:.4f} |"
              f"G Loss: {g_loss:.4f}"
        )


Epoch: 0 | D_loss: 0.5187 |G Loss: 1.7224
Epoch: 1 | D_loss: 0.5072 |G Loss: 2.9345
Epoch: 2 | D_loss: 0.4794 |G Loss: 3.8785
Epoch: 3 | D_loss: 0.4222 |G Loss: 4.5273
Epoch: 4 | D_loss: 0.3620 |G Loss: 4.9568
Epoch: 5 | D_loss: 0.2854 |G Loss: 5.1818
Epoch: 6 | D_loss: 0.2218 |G Loss: 5.2999
Epoch: 7 | D_loss: 0.1845 |G Loss: 5.3196
Epoch: 8 | D_loss: 0.1479 |G Loss: 5.3041
Epoch: 9 | D_loss: 0.1223 |G Loss: 5.2326
Epoch: 10 | D_loss: 0.0911 |G Loss: 5.2073
Epoch: 11 | D_loss: 0.0719 |G Loss: 5.1816
Epoch: 12 | D_loss: 0.0550 |G Loss: 5.1924
Epoch: 13 | D_loss: 0.0530 |G Loss: 5.2393
Epoch: 14 | D_loss: 0.0382 |G Loss: 5.2414
Epoch: 15 | D_loss: 0.0399 |G Loss: 5.2749
Epoch: 16 | D_loss: 0.0298 |G Loss: 5.2944
Epoch: 17 | D_loss: 0.0338 |G Loss: 5.3060
Epoch: 18 | D_loss: 0.0262 |G Loss: 5.3219
Epoch: 19 | D_loss: 0.0191 |G Loss: 5.2529
Epoch: 20 | D_loss: 0.0252 |G Loss: 5.2939
Epoch: 21 | D_loss: 0.0187 |G Loss: 5.2868
Epoch: 22 | D_loss: 0.0192 |G Loss: 5.2722
Epoch: 23 | D_loss: 0

In [ ]:

#Generate Images

noise = np.random.normal(
    0,
    1,
    (10, latent_dim)
)

generated_images = generator.predict(
    noise,
    verbose=0
)

60000

In [ ]:
#Display Generated Images

plt.figure(figsize=(10, 2))

for i in range(10):

    plt.subplot(1, 10, i + 1)

    plt.imshow(
        generated_images[i].reshape(28, 28),
        cmap="gray"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()